In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# قراءة الملف مع حل مشكلة الرموز (Encoding)
df = pd.read_csv('data.csv', encoding='ISO-8859-1')

# عرض أول 5 أسطر للتأكد
df.head()







,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [13]:
# 1. معرفة ملخص عن الأعمدة وأنواعها
print("--- معلومات عامة عن البيانات ---")
df.info()

# 2. حساب عدد القيم المفقودة في كل عمود
print("\n--- عدد القيم المفقودة ---")
print(df.isnull().sum())

--- معلومات عامة عن البيانات ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB

--- عدد القيم المفقودة ---
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [14]:
# 1. حذف السطور التي ليس لها رقم عميل (لأننا لا نستطيع تحليل عملاء مجهولين)
df.dropna(subset=['CustomerID'], inplace=True)

# 2. تحويل عمود التاريخ من نص إلى تاريخ حقيقي (Datetime)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 3. حساب إجمالي المبلغ لكل عملية شراء (الكمية × سعر الوحدة)
df['TotalSum'] = df['Quantity'] * df['UnitPrice']

# التأكد من النتائج
print("عدد السطور المتبقية بعد الحذف:", len(df))
df.head()

عدد السطور المتبقية بعد الحذف: 406829


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSum
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [15]:
# 1. تحديد تاريخ "اليوم" (سنفترضه بعد آخر عملية شراء بيوم واحد للحساب)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. تجميع البيانات حسب كل عميل (CustomerID)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: كم يوم مضى على آخر شراء
    'InvoiceNo': 'count',                                   # Frequency: كم عدد العمليات
    'TotalSum': 'sum'                                       # Monetary: إجمالي المبالغ
})

# 3. إعادة تسمية الأعمدة لتكون واضحة
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

# عرض أول 5 عملاء بعد الحساب
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,2,0.00
12347.0,2,182,4310.00
12348.0,75,31,1797.24
12349.0,19,73,1757.55
12350.0,310,17,334.40


In [16]:
# تقسيم العملاء إلى 5 مجموعات لكل مقياس
# بالنسبة للـ Recency: الرقم الأقل هو الأفضل (لذلك نضع الترتيب عكسي False)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# بالنسبة للـ Frequency و Monetary: الرقم الأعلى هو الأفضل
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# دمج الدرجات في رقم واحد يسمى RFM_Score
rfm['RFM_Group'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Group
CustomerID,,,,,,,
12346.0,326,2,0.00,1,1,1,111
12347.0,2,182,4310.00,5,5,5,555
12348.0,75,31,1797.24,2,3,4,234
12349.0,19,73,1757.55,4,4,4,444
12350.0,310,17,334.40,1,2,2,122


In [17]:
# خريطة التصنيف بناءً على درجات R و F
segs = {
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At Risk',
    r'[1-2]5': 'Can\'t Loose Them',
    r'3[1-2]': 'About to Sleep',
    r'33': 'Need Attention',
    r'[3-4][4-5]': 'Loyal Customers',
    r'41': 'Promising',
    r'51': 'New Customers',
    r'[4-5][2-3]': 'Potential Loyalists',
    r'5[4-5]': 'Champions'
}

# تطبيق التصنيف على الجدول
rfm['Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str)
rfm['Segment'] = rfm['Segment'].replace(segs, regex=True)

# عرض النتيجة النهائية
rfm.head()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Group,Segment
CustomerID,,,,,,,,
12346.0,326,2,0.00,1,1,1,111,Hibernating
12347.0,2,182,4310.00,5,5,5,555,Champions
12348.0,75,31,1797.24,2,3,4,234,At Risk
12349.0,19,73,1757.55,4,4,4,444,Loyal Customers
12350.0,310,17,334.40,1,2,2,122,Hibernating


In [18]:
# حفظ النتيجة في ملف Excel
rfm.to_excel('Customer_Segmentation_Results.xlsx')
print("✅ تم حفظ النتائج بنجاح في ملف Excel!")

✅ تم حفظ النتائج بنجاح في ملف Excel!
